In [33]:
import gc
import json
import os
import shutil
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

CHUNKS_FILE = "chunks.json"
CHROMA_DB_DIR = "./chroma_db"
COLLECTION_NAME = "academic_papers"

# Load Text Chunks


In [29]:
if not os.path.exists(CHUNKS_FILE):
    print(f"Error: {CHUNKS_FILE} not found. Please run injestion.ipynb first!")
else:
    with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    if not chunks:
        print("No chunks found to index.")
    else:
        print(f"Loaded {len(chunks)} chunks from {CHUNKS_FILE}")

Loaded 526 chunks from chunks.json


# Initialize Embeddings Model


In [30]:
print("Loading HuggingFace Embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"local_files_only": True}
)

Loading HuggingFace Embeddings model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2568.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Prepare Document Objects


In [31]:
def infer_subsection_from_section(section_value):
    text = str(section_value or "").strip()
    if ":" not in text:
        return text or "Main Text", None

    left, right = text.split(":", 1)
    left = left.strip()
    right = right.strip()
    if left and right:
        return left, right
    return text or "Main Text", None



def ensure_hierarchy_metadata(raw_metadata):
    metadata = dict(raw_metadata or {})
    level_1 = metadata.get("path_level_1") or metadata.get("title") or metadata.get("source") or "Unknown Document"
    level_2 = metadata.get("path_level_2") or metadata.get("section") or "Main Text"
    level_3 = metadata.get("path_level_3") or metadata.get("subsection")

    inferred_section, inferred_subsection = infer_subsection_from_section(level_2)
    level_2 = inferred_section or "Main Text"
    if not level_3:
        level_3 = inferred_subsection

    # Keep a 3-level schema for every chunk.
    level_3 = str(level_3).strip() if level_3 else "General"

    metadata["section"] = str(level_2)
    metadata["subsection"] = level_3
    metadata["path_level_1"] = str(level_1)
    metadata["path_level_2"] = str(level_2)
    metadata["path_level_3"] = level_3
    metadata["hierarchy_path"] = f"{metadata['path_level_1']} -> {metadata['path_level_2']} -> {metadata['path_level_3']}"

    try:
        metadata["path_depth"] = int(metadata.get("path_depth", 3))
    except (TypeError, ValueError):
        metadata["path_depth"] = 3

    metadata["path_depth"] = 3
    return metadata



documents = []
if 'chunks' in locals() and chunks:
    for chunk in chunks:
        # Safely extract the text
        content = chunk.get("text")

        # Skip if the content is missing or empty.
        if not content or not str(content).strip():
            continue

        metadata = ensure_hierarchy_metadata(chunk.get("metadata", {}))
        doc = Document(
            page_content=str(content),
            metadata=metadata,
        )
        documents.append(doc)

    print(f" Prepared {len(documents)} clean documents for embeddings.")

 Prepared 526 clean documents for embeddings.


# Chroma Vector Store


In [34]:
if documents:
    db_dir = os.path.abspath(CHROMA_DB_DIR)
    print(f"Initializing Chroma DB at {db_dir} and embedding all chunks...")

    # Release possible existing handles before resetting the folder.
    if "vectorstore" in locals():
        del vectorstore
    gc.collect()

    # Reset the existing DB folder so repeated runs do not create duplicate vectors.
    if os.path.exists(db_dir):
        if os.path.isdir(db_dir):
            shutil.rmtree(db_dir)
        else:
            os.remove(db_dir)
    os.makedirs(db_dir, exist_ok=True)

    # Store in chroma
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=db_dir,
    )

    print("\nDone: All chunks embedded and saved to ChromaDB.")
else:
    print("No documents to store.")

Initializing Chroma DB at /home/wiz/Documents/hrag/chroma_db and embedding all chunks...

Done: All chunks embedded and saved to ChromaDB.


# Sanity Check Query


In [35]:
if 'vectorstore' in locals():
    print("-" * 50)
    print("Running a quick sanity check query with metadata filtering...")
    
    test_query = "What is the focus of this paper?"
    test_filter = {"year": {"$eq": 2024}} # Testing our mapped metadata
    
    try:
        results = vectorstore.similarity_search(test_query, k=1, filter=test_filter)
        if results:
            print("\nFound result successfully matching year=2024")
            print(f"Source: {results[0].metadata.get('source')}")
            print(f"Content preview: {results[0].page_content[:150]}...")
        else:
            print("No results found for year=2024. Is the p2.pdf data parsed correctly?")
    except Exception as e:
        print(f"Query check failed: {e}")

--------------------------------------------------
Running a quick sanity check query with metadata filtering...

Found result successfully matching year=2024
Source: p2.pdf
Content preview: [40] S. Min, M. Lewis, L. Zettlemoyer, and H. Hajishirzi. Metaicl: Learning to learn in context. arXiv preprint arXiv:2110.15943, 2021....
